# Results 4 — Variant-level pleiotropy

Cluster counts, the negative-binomial model of the variant pleiotropy score (vPS), the
directionality analysis, and the data behind Figure 3.

| file | panel |
| --- | --- |
| `plot_a.csv` | observed and predicted vPS per MAF bin (Figure 3a) |
| `plot_b.csv` | univariate and joint model coefficients (Figure 3b) |
| `figure_3_apoe.csv` | every disease association of the two APOE variants (Figure 3c) |

Predicted power assumes the variant acts on most traits at an effect size an order of
magnitude below its largest observed effect: the non-centrality parameter is
`maxAbsBeta^2 * maxEffectiveSampleSize * maxVarG / 11`, and power is the survival function of
a non-central chi-square at the genome-wide threshold.

In [ ]:
import collections

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import ncx2, spearmanr

from manuscript_methods import clusters, paper

numbers = {}

COVARIATES = [
    "maxAbsBetaNormalised",
    "maxMAFNormalised",
    "maxEffectiveSampleSizeNormalised",
    "gerpNormalisedNormalised",
    "vepBinaryNormalised",
    "predictedPowerNormalised",
]
LABELS = {
    "maxAbsBetaNormalised": "Absolute beta",
    "maxMAFNormalised": "MAF",
    "maxEffectiveSampleSizeNormalised": "Sample size",
    "gerpNormalisedNormalised": "GERP",
    "vepBinaryNormalised": "PAV",
    "predictedPowerNormalised": "Predicted power",
}
MAF_BINS = [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
MAF_BIN_LABELS = ["0-0.01", "0.01-0.05", "0.05-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5"]
GENOME_WIDE_CHI2 = 32.84125  # chi-square value corresponding to p = 5e-8 with 1 degree of freedom

## Clusters

In [ ]:
cluster_table = pd.read_parquet(paper.derived("variant_clusters"))

numbers["R4.01"] = len(cluster_table)
numbers["R4.02"] = int((cluster_table["uniqueLeadVariants"] > 1).sum())
numbers["R4.03"] = int((cluster_table["uniqueDiseases"] > 1).sum())
numbers["R4.04"] = int(cluster_table["uniqueDiseases"].max())
numbers["R4.05"] = round(float(cluster_table["uniqueDiseases"].mean()), 2)
numbers["R4.06"] = int((cluster_table["uniqueTherapeuticAreas"] > 1).sum())
numbers["R4.07"] = int(cluster_table["uniqueTherapeuticAreas"].max())
numbers["R4.08"] = round(float(cluster_table["uniqueTherapeuticAreas"].mean()), 2)
rho, pvalue = spearmanr(cluster_table["uniqueDiseases"], cluster_table["uniqueTherapeuticAreas"])
numbers["R4.09"] = round(float(rho), 2)
print({k: numbers[k] for k in sorted(numbers)}, "| Spearman P:", pvalue)

## Cluster-level covariates

Each cluster is represented by one lead variant: the one associated with most diseases, ties
broken by variant id.

In [ ]:
credible_sets = clusters.load_credible_sets()
edges = clusters.load_edges(set(credible_sets["studyLocusId"]))
components = clusters.cluster(list(zip(credible_sets["studyLocusId"], credible_sets["variantId"])), edges)

locus_variant = dict(zip(credible_sets["studyLocusId"], credible_sets["variantId"]))
locus_traits = dict(zip(credible_sets["studyLocusId"], credible_sets["diseaseIds"]))

features = pd.read_parquet(paper.derived("variant_features"))[
    ["variantId", "maxAbsBeta", "maxMAF", "maxEffectiveSampleSize", "maxVarG", "gerpNormalised", "vepScore"]
].drop_duplicates("variantId")

rows = []
for _, members in components:
    per_variant = collections.defaultdict(set)
    for locus_id in members:
        per_variant.setdefault(locus_variant[locus_id], set())
        traits = locus_traits.get(locus_id)
        if traits is not None:
            per_variant[locus_variant[locus_id]].update(traits)
    representative = sorted(per_variant.items(), key=lambda kv: (-len(kv[1]), kv[0]))[0][0]
    all_traits = set().union(*per_variant.values()) if per_variant else set()
    rows.append({"clusterSize": len(members), "vPS": len(all_traits), "clusterVariantId": representative})

frame = pd.DataFrame(rows).merge(features, left_on="clusterVariantId", right_on="variantId", how="inner")
print("clusters joined to variant features:", len(frame), "of", len(components))

In [ ]:
frame["ncp"] = (frame["maxAbsBeta"] ** 2 * frame["maxEffectiveSampleSize"] * frame["maxVarG"]) / 11
frame["predictedPower"] = ncx2.sf(x=GENOME_WIDE_CHI2, df=1, nc=frame["ncp"])
frame["gerpNormalised"] = frame["gerpNormalised"].fillna(frame["gerpNormalised"].mean())
frame["vepBinary"] = (frame["vepScore"] >= 0.66).astype(int)

# Every covariate is min-max scaled so the coefficients are comparable in the forest plot.
for column in ["maxAbsBeta", "maxMAF", "gerpNormalised", "vepBinary", "maxEffectiveSampleSize", "predictedPower"]:
    span = frame[column].max() - frame[column].min()
    frame[f"{column}Normalised"] = 0.0 if span == 0 else (frame[column] - frame[column].min()) / span

## Figure 3b — negative binomial models

In [ ]:
def fit(covariates):
    """Negative binomial fit of vPS on the given covariates."""
    x = sm.add_constant(frame[covariates].copy())
    model = sm.NegativeBinomial(frame["vPS"], x).fit(disp=False, maxiter=1000)
    return model, x


records = []
for covariate in COVARIATES:
    model, _ = fit([covariate])
    ci = model.conf_int()
    records.append(
        {
            "covariate": covariate,
            "model_type": "Univariate",
            "coefficient": model.params[covariate],
            "std_error": model.bse[covariate],
            "p_value": model.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

joint, x_joint = fit(COVARIATES)
ci = joint.conf_int()
for covariate in COVARIATES:
    records.append(
        {
            "covariate": covariate,
            "model_type": "Multi",
            "coefficient": joint.params[covariate],
            "std_error": joint.bse[covariate],
            "p_value": joint.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

coefficients = pd.DataFrame(records)
coefficients["covariate_label"] = coefficients["covariate"].map(LABELS)
coefficients["y_numerical"] = coefficients["covariate"].map({c: i for i, c in enumerate(COVARIATES)})
coefficients["y_plot"] = coefficients["y_numerical"] + np.where(coefficients["model_type"] == "Univariate", -0.1, 0.1)
coefficients = coefficients[
    [
        "covariate",
        "covariate_label",
        "model_type",
        "coefficient",
        "std_error",
        "p_value",
        "ci_lower",
        "ci_upper",
        "y_numerical",
        "y_plot",
    ]
]
coefficients.to_csv(paper.derived("plot_b.csv"), index=False)
coefficients.round(4)

## Variance explained

In [ ]:
without_power = [c for c in COVARIATES if c != "predictedPowerNormalised"]
joint_no_power, x_no_power = fit(without_power)
power_only, x_power = fit(["predictedPowerNormalised"])
sample_size_only, x_sample = fit(["maxEffectiveSampleSizeNormalised"])


def r2(model, x):
    """Squared Pearson correlation between observed and predicted vPS."""
    return float(np.corrcoef(frame["vPS"], model.predict(x))[0, 1] ** 2)


numbers["R4.10"] = round(100 * r2(power_only, x_power), 1)
numbers["R4.11"] = round(100 * r2(joint, x_joint), 1)
numbers["R4.12"] = round(100 * r2(joint_no_power, x_no_power), 1)
numbers["R4.13"] = round(100 * r2(sample_size_only, x_sample), 2)
print({k: numbers[k] for k in ["R4.10", "R4.11", "R4.12", "R4.13"]})

## Figure 3a — observed and predicted vPS per MAF bin

In [ ]:
binned = frame.copy()
binned["predicted_traits_full_model"] = joint.predict(x_joint)
binned["predicted_traits_no_power"] = joint_no_power.predict(x_no_power)
binned["maxMAF_bin"] = pd.cut(binned["maxMAF"], bins=MAF_BINS, labels=MAF_BIN_LABELS, right=False)

bins = (
    binned.groupby("maxMAF_bin", observed=False)
    .agg(
        observed_mean=("vPS", "mean"),
        observed_sem=("vPS", "sem"),
        predicted_full_mean=("predicted_traits_full_model", "mean"),
        predicted_full_sem=("predicted_traits_full_model", "sem"),
        predicted_no_power_mean=("predicted_traits_no_power", "mean"),
        predicted_no_power_sem=("predicted_traits_no_power", "sem"),
    )
    .reset_index()
)
bins["maxMAF_bin"] = bins["maxMAF_bin"].astype(str)
bins.to_csv(paper.derived("plot_a.csv"), index=False)
bins.round(4)

## Directionality

Concordance is computed on the minor allele, over lead variants associated with more than one
disease and carrying a signed effect.

In [ ]:
variant_features = pd.read_parquet(paper.derived("variant_features"))
pleiotropic = variant_features[
    (variant_features["uniqueDiseases"] > 1) & variant_features["betaSignConcordance"].notna()
]

numbers["R4.14"] = len(pleiotropic)
numbers["R4.15"] = int((pleiotropic["betaSignConcordance"] == 1.0).sum())
numbers["R4.16"] = int((pleiotropic["betaSignConcordance"] < 1.0).sum())

highly_pleiotropic = pleiotropic[pleiotropic["uniqueDiseases"] >= 10]
discordant = highly_pleiotropic[highly_pleiotropic["betaSignConcordance"] <= 0.8]
numbers["R4.17"] = len(highly_pleiotropic)
numbers["R4.18"] = len(discordant)
numbers["R4.19"] = len({gene for genes in discordant["prioritisedGenes"] for gene in (genes or [])})
print({k: numbers[k] for k in ["R4.14", "R4.15", "R4.16", "R4.17", "R4.18", "R4.19"]})
print("fully concordant share: %.1f%%" % (100 * numbers["R4.15"] / numbers["R4.14"]))

In [ ]:
# Supplementary Table 2: the discordant highly pleiotropic lead variants.
st2 = discordant.sort_values("uniqueDiseases", ascending=False)[
    ["variantId", "prioritisedGenes", "uniqueDiseases", "uniqueTherapeuticAreas", "betaSignConcordance", "maxMAF"]
]
st2.to_csv(paper.derived("st2_discordant_variants.csv"), index=False)
st2.head(10)

## The two APOE variants of Figure 3c

In [ ]:
APOE_VARIANTS = ["19_44908684_T_C", "19_44908822_C_T"]

apoe = variant_features[variant_features["variantId"].isin(APOE_VARIANTS)].set_index("variantId")
numbers["R4.20"] = int(apoe.loc["19_44908684_T_C", "uniqueDiseases"])
numbers["R4.21"] = round(float(apoe.loc["19_44908684_T_C", "betaSignConcordance"]), 2)
numbers["R4.22"] = int(apoe.loc["19_44908684_T_C", "uniqueTherapeuticAreas"])
print({k: numbers[k] for k in ["R4.20", "R4.21", "R4.22"]})

In [ ]:
import pyarrow.compute as pc
import pyarrow.dataset as ds

names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

associations = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(
        columns={
            "variantId": ds.field("variantId"),
            "studyId": ds.field("studyId"),
            "diseaseIds": ds.field("diseaseIds"),
            "originalBeta": ds.field("originalBeta"),
            "estimatedBeta": pc.struct_field(ds.field("rescaledStatistics"), "minorAlleleEstimatedBeta"),
            "pValueMantissa": pc.struct_field(ds.field("variantStatistics"), "pValueMantissa"),
            "pValueExponent": pc.struct_field(ds.field("variantStatistics"), "pValueExponent"),
        },
        filter=pc.field("variantId").isin(APOE_VARIANTS),
    )
    .to_pandas()
)
associations = associations[associations["originalBeta"].notna()].copy()
associations["diseaseNames"] = associations["diseaseIds"].map(lambda ids: [names.get(d) for d in (ids or [])])
associations["mappedTherapeuticAreas"] = associations["diseaseIds"].map(
    lambda ids: sorted({areas.get(d, "other") for d in (ids or [])})
)
associations["therapeuticAreaNames"] = associations["mappedTherapeuticAreas"].map(
    lambda tas: [paper.THERAPEUTIC_AREAS.get(t, "other") for t in tas]
)
associations["neg_log10_p"] = -(np.log10(associations["pValueMantissa"]) + associations["pValueExponent"])

columns = [
    "studyId",
    "diseaseIds",
    "diseaseNames",
    "mappedTherapeuticAreas",
    "therapeuticAreaNames",
    "estimatedBeta",
    "pValueMantissa",
    "pValueExponent",
    "neg_log10_p",
]
outputs = ["variant_pleiotropy_data_exploded.csv", "variant_pleiotropy_data_exploded_2.csv"]
for variant, name in zip(APOE_VARIANTS, outputs):
    subset = associations[associations["variantId"] == variant].explode("therapeuticAreaNames")
    subset[columns].to_csv(paper.derived(name), index=False)
    print(name, subset.shape)

## Numbers

In [ ]:
print(paper.save_results("variant_pleiotropy", numbers))
pd.Series(numbers).to_frame("computed")